# 🛠️ Project Requirements & Dependencies

This project is optimized for real-time computer vision inference leveraging **NVIDIA RTX 4060 GPU** hardware acceleration.

### Core Dependencies:
* **Deep Learning & Computer Vision:** `torch`, `torchvision` (CUDA compiled), `ultralytics` (YOLOv8), `opencv-python`
* **Data Engine:** `numpy`
* **Parallel Processing:** `threading`

In [ ]:
# =====================================================================
# 🛠️ CELL 1 — CORE ENVIRONMENT SETUP & HARDWARE COMPUTE
# =====================================================================
import os  # Import the operating system module for environment configurations
import json  # Import JSON module to read/write system configuration and telemetry files
import time  # Import time module to handle benchmarks, tracking, and timeouts
import cv2  # Import OpenCV library for advanced image and video processing tasks
import torch  # Import PyTorch deep learning framework for GPU compute allocation
import numpy as np  # Import NumPy for fast numerical and matrix operations on arrays
from threading import Thread  # Import Thread to run the stream reader in a parallel background process
from ultralytics import YOLO  # Import YOLO from ultralytics to load and run object detection models

# Prevent initialization crashes related to duplicate OpenMP libraries inside the notebook runtime
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Evaluate hardware ecosystem to check if a compatible Nvidia CUDA GPU is available on the local machine
if torch.cuda.is_available():
    # Loop through available devices to explicitly target the NVIDIA RTX 4060 hardware profile
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        if "4060" in device_name or "RTX" in device_name.upper():
            torch.cuda.set_device(i)  # Set the active device index to the RTX 4060 GPU
            print(f"[INFO] Initial hardware acceleration environment configured successfully on device: {device_name}")
            break
else:
    print("[WARNING] CUDA GPU acceleration not detected. Falling back to default CPU processing core.")

[INFO] Initial hardware acceleration environment configured successfully on device: NVIDIA GeForce RTX 4060


# 📡 Core Stream Classes & Telemetry Utilities

In [ ]:
# =====================================================================
# 📡 CELL 2 — VIDEO STREAM CONTROLLER & DATA PIPELINE METRICS
# =====================================================================
class RTSPVideoStream:
    """Manages continuous high-speed video frame extraction inside a background worker thread."""
    def __init__(self, src):
        # Initialize video capture object with FFMPEG backend wrapper configurations
        self.stream = cv2.VideoCapture(src, cv2.CAP_FFMPEG)
        self.stream.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Force hardware buffer stack depth to 1 to eliminate frame latency
        self.stopped = False  # Main thread lifecycle state indicator flag
        self.ret = False  # Boolean frame capture confirmation status
        self.frame = None  # Storage array for the latest extracted image matrix

    def start(self):
        """Spawns and kicks off the background daemon worker thread process."""
        t = Thread(target=self.update, args=())
        t.daemon = True  # Thread automatically dies when the parent application process closes
        t.start()
        return self

    def update(self):
        """Continuous frame grab loop executing entirely in the background thread layer."""
        while True:
            if self.stopped:
                return
            self.ret, self.frame = self.stream.read()
            if not self.ret:
                self.stopped = True

    def read(self):
        """Retrieves the most recent frame array fetched by the worker thread."""
        return self.ret, self.frame

    def stop(self):
        """Safely stops frame capture and releases heavy system hardware resources."""
        self.stopped = True
        self.stream.release()

def load_dashboard_config():
    """Reads real-time dynamic configurations from the local json dashboard setup file."""
    config_path = "config.json"
    if os.path.exists(config_path):
        try:
            with open(config_path, "r") as f:
                return json.load(f)
        except Exception as e:
            print(f"[ERROR] Failed to read dynamic dashboard JSON configuration parameters: {e}")
    # Default fallback ratio metrics if local configuration file does not exist
    return {"counting_line_ratio": 0.50, "zones": []}

def write_telemetry(total_in, total_out, inside, fps):
    """Outputs current tracking and analytics metadata into localized json structures."""
    telemetry_path = "telemetry.json"
    data = {
        "total_in": total_in,
        "total_out": total_out,
        "current_inside": inside,
        "fps": round(fps, 1)
    }
    try:
        with open(telemetry_path, "w") as f:
            json.dump(data, f)
    except Exception as e:
        pass

# 🧠 YOLO Deep Learning Weights Initialization

In [ ]:
# =====================================================================
# 🧠 CELL 3 — MODEL ARCHITECTURE LOADING
# =====================================================================
# Load the unified deep learning computer vision weights onto the active runtime environment
model = YOLO("yolo26l.pt", task="detect")
print("[SUCCESS] Deep learning computer vision neural network weights initialized successfully.")

[SUCCESS] Deep learning computer vision neural network weights initialized successfully.


: 

# 📊 Spatial Hybrid Counting & Advanced Logic Engine

This section isolates the mathematical line-crossing logic, visual bounding boxes drawing, state identification, and high-precision duration trackers.

In [ ]:
# =====================================================================
# 📊 CELL 4 — SPATIAL MATHEMATICS & HIGH-PRECISION COMPUTER VISION LOOP
# =====================================================================

# Global counter metrics and state tracking dictionaries initialization
total_in = 0  # Running sum total of objects crossing entry thresholds
total_out = 0  # Running sum total of objects crossing exit thresholds
track_history = {}  # Stores historical horizontal path coordinate trails for direction analysis
crossed_direction = {}  # Keeps track of the current cross status for active unique track IDs
entry_timestamps = {}  # Records exact system epoch time when a tracking ID changes its state
dwell_durations = {}  # Calculates total accumulated active seconds inside the CURRENT state per ID
completed_inside_times = []  # Persistent registry to log completed stay durations for global average calculation

# Define connection target parameters and desktop graphics frame dimensions
udp_url = "udp://0.0.0.0:9999?listen&timeout=2000000"
window_name = "LIVE VIDEO ANALYSIS ACCELERATION INTERFACE"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

frame_count = 0  # Counter track to orchestrate timed configuration polling intervals
user_requested_exit = False  # Graceful teardown condition flag

while True:
    if user_requested_exit:
        break
        
    print(f"[INFO] Initializing connection to pipeline target feed channel: {udp_url}")
    vstream = RTSPVideoStream(udp_url).start()
    time.sleep(1.0)  # Allow sufficient buffer space for stream setup and validation pipelines

    if not vstream.ret:
        print("[ERROR] Connection pipeline failed to capture data packets. Re-attempting connection link in 3 seconds...")
        vstream.stop()
        time.sleep(3)
        continue

    print("[SUCCESS] Network link established! Processing active frames. Press 'q' to abort execution.")
    config = load_dashboard_config()
    last_packet_time = time.time()

    while True:
        start_benchmark_time = time.time()
        frame_count += 1

        # Synchronize analytics guidelines with dashboard configurations every 10 frames
        if frame_count % 10 == 0:
            config = load_dashboard_config()

        ret, frame = vstream.read()
        if not ret or frame is None:
            if time.time() - last_packet_time > 3.0:
                print("[WARNING] Connection timeout reached on streaming socket channels. Retrying network link...")
                vstream.stop()
                break
            time.sleep(0.001)
            continue

        last_packet_time = time.time()
        h, w, _ = frame.shape

        # ####################################################################################################
        # 🔻🔻🔻 [LINE POSITION & CONFIGURATION RECAP BLOCK] 🔻🔻🔻
        # ####################################################################################################
        counting_line_ratio = config.get("counting_line_ratio", 0.50)  # Read ratio position boundary marker value
        line_x_pixel = int(w * counting_line_ratio)  # Calculate actual pixel coordinates based on frame width
        
        # Render the tracking boundary reference frame onto the active display canvas
        cv2.line(frame, (line_x_pixel, 0), (line_x_pixel, h), (0, 0, 255), 3)
        # ####################################################################################################
        # 🔺🔺🔺 [LINE POSITION & CONFIGURATION RECAP BLOCK] 🔺🔺🔺
        # ####################################################################################################

        # Execute object tracking pipeline with hyperparameter tuning configurations
        results = model.track(frame, imgsz=640, conf=0.25, tracker="bytetrack.yaml", persist=True, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()  # Local bounding box coordinate matrix array
            ids = results[0].boxes.id.cpu().numpy().astype(int)  # Unique track identity identifier numbers
            clses = results[0].boxes.cls.cpu().numpy().astype(int)  # Classification category index flags

            current_epoch_time = time.time()

            for box, track_id, cls_id in zip(boxes, ids, clses):
                # Calculate coordinates for center point tracking metrics
                center_x = int((box[0] + box[2]) / 2)
                center_y = int((box[1] + box[3]) / 2)

                # Initialize timeline tracking indicators if this is a newly assigned identifier
                if track_id not in entry_timestamps:
                    entry_timestamps[track_id] = current_epoch_time
                
                # #################################################################################
                # 🔻🔻🔻 [MATH DIRECTION LOGIC CROSSING & TIME RESET BLOCK] 🔻🔻🔻
                # #################################################################################
                historical_x = track_history.get(track_id, center_x)
                state_changed = False  # Track if a crossing event happens to reset timers

                # Vector direction tracking evaluation logic formulas
                if historical_x <= line_x_pixel and center_x > line_x_pixel and crossed_direction.get(track_id) != "IN":
                    total_in += 1
                    crossed_direction[track_id] = "IN"
                    state_changed = True  # Signal to reset timer from the beginning for the new IN state
                elif historical_x >= line_x_pixel and center_x < line_x_pixel and crossed_direction.get(track_id) != "OUT":
                    total_out += 1
                    
                    # PERSISTENT SESSION TIME CAPTURE: Log completed inside stay before resetting timer
                    if crossed_direction.get(track_id) == "IN" and track_id in dwell_durations:
                        completed_inside_times.append(dwell_durations[track_id])
                        
                    crossed_direction[track_id] = "OUT"
                    state_changed = True  # Signal to reset timer from the beginning for the new OUT state

                # If state changed, overwrite baseline timestamp to begin calculation from zero
                if state_changed:
                    entry_timestamps[track_id] = current_epoch_time

                track_history[track_id] = center_x  # Store current center location to check on the next frame
                # #################################################################################
                # 🔺🔺🔺 [MATH DIRECTION LOGIC CROSSING & TIME RESET BLOCK] 🔺🔺🔺
                # #################################################################################

                # Update current active timeline tracking benchmarks relative to the current active state session
                elapsed_seconds = current_epoch_time - entry_timestamps[track_id]
                dwell_durations[track_id] = elapsed_seconds

                # --- CONDITIONAL VISUAL RENDER BLOCK: ONLY DRAW RED, GREEN, PURPLE ---
                draw_bounding_box = True  # Status flag to verify if drawing needs to execute
                
                if cls_id == 1:
                    bounding_color = (255, 0, 255)  # Purple layout canvas represents Employees
                    role_label_string = "EMP"
                elif crossed_direction.get(track_id) == "IN":
                    bounding_color = (0, 255, 0)  # Green layout canvas represents Customers inside
                    role_label_string = "IN"
                elif crossed_direction.get(track_id) == "OUT":
                    bounding_color = (0, 0, 255)  # Red layout canvas represents Customers exiting
                    role_label_string = "OUT"
                else:
                    draw_bounding_box = False  # Completely suppress rendering for default uncrossed tracks (No light blue box)

                # Execute rendering tasks only if the target matches one of our three core approved states
                if draw_bounding_box:
                    cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), bounding_color, 2)
                    text_label_tag = f"{role_label_string} #{track_id} | {int(elapsed_seconds)}s"
                    cv2.putText(frame, text_label_tag, (int(box[0]), int(box[1]) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, bounding_color, 2)

        # Periodic dictionary pruning routine to keep system RAM lean and fast
        if len(track_history) > 300:
            historical_keys_list = list(track_history.keys())[:-100]
            for targeted_key in historical_keys_list:
                track_history.pop(targeted_key, None)
                crossed_direction.pop(targeted_key, None)
                entry_timestamps.pop(targeted_key, None)
                dwell_durations.pop(targeted_key, None)

        # Net metrics monitoring calculation configurations
        net_inside_count = max(0, total_in - total_out)
        fps = 1.0 / max(time.time() - start_benchmark_time, 0.001)

        # Output current execution snapshot telemetry to local filesystem storage paths
        write_telemetry(total_in, total_out, net_inside_count, fps)

        # Calculate live average from the persistent registry of historical completed customer sessions
        avg_time = sum(completed_inside_times) / len(completed_inside_times) if completed_inside_times else 0.0

        # UI Overlay Construction
        overlay = frame.copy()  
        cv2.rectangle(overlay, (0, 0), (w, 55), (20, 20, 20), -1)  
        cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)  

        # Format header text with the targeted Inside Average Time metric
        header_text = f"IN: {total_in} | OUT: {total_out} | INSIDE: {net_inside_count} | AVG TIME: {int(avg_time)}s | {round(fps, 1)} FPS"
        cv2.putText(frame, header_text, (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)  

        cv2.imshow(window_name, frame)  
        
        if cv2.waitKey(1) & 0xFF == ord('q'):  
            print("[INFO] 'q' key pressed. Shutting down...")  
            vstream.stop()  
            cv2.destroyAllWindows()  
            user_requested_exit = True  
            break

[INFO] Initializing connection to pipeline target feed channel: udp://0.0.0.0:9999?listen&timeout=2000000
[ERROR] Connection pipeline failed to capture data packets. Re-attempting connection link in 3 seconds...
[INFO] Initializing connection to pipeline target feed channel: udp://0.0.0.0:9999?listen&timeout=2000000
[SUCCESS] Network link established! Processing active frames. Press 'q' to abort execution.
[WARNING] Connection timeout reached on streaming socket channels. Retrying network link...
[INFO] Initializing connection to pipeline target feed channel: udp://0.0.0.0:9999?listen&timeout=2000000
[ERROR] Connection pipeline failed to capture data packets. Re-attempting connection link in 3 seconds...
[INFO] Initializing connection to pipeline target feed channel: udp://0.0.0.0:9999?listen&timeout=2000000
[ERROR] Connection pipeline failed to capture data packets. Re-attempting connection link in 3 seconds...
[INFO] Initializing connection to pipeline target feed channel: udp://0.0.